# Projekt CNN — Oxford-IIIT Pet Dataset (v2)

Ten notebook spełnia wymagania projektu:
- analiza datasetu,
- działająca sieć CNN,
- 3 eksperymenty (każdy kolejny ulepsza poprzedni),
- wyniki numeryczne i wizualne,
- wnioski.

Dodatkowo zapisuje artefakty do `artifacts_v2/` dla frontendu `frontend_app_v2.py`.


In [ ]:
import json
import random
import time
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Definicje modeli (inline, bez zależności od innych plików)
class BaselineCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class ImprovedCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def create_transfer_model(num_classes: int, freeze_backbone: bool = False):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.fc.in_features, num_classes),
    )
    return model


def build_model_from_key(model_key: str, num_classes: int):
    if model_key == "baseline":
        return BaselineCNN(num_classes)
    if model_key == "improved":
        return ImprovedCNN(num_classes)
    if model_key == "transfer":
        return create_transfer_model(num_classes, freeze_backbone=False)
    raise ValueError(model_key)


In [ ]:
# Dane i analiza datasetu
DATA_ROOT = Path("data")
ARTIFACTS_ROOT = Path("artifacts_v2")
ARTIFACTS_ROOT.mkdir(exist_ok=True)

IMAGE_SIZE = (128, 128)
NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()

trainval_raw = datasets.OxfordIIITPet(
    root=DATA_ROOT,
    split="trainval",
    target_types="category",
    download=True,
)
test_raw = datasets.OxfordIIITPet(
    root=DATA_ROOT,
    split="test",
    target_types="category",
    download=True,
)

CLASS_NAMES = trainval_raw.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Liczba klas:", NUM_CLASSES)
print("Trainval:", len(trainval_raw), "Test:", len(test_raw))

labels = [label for _, label in trainval_raw]
class_counts = pd.Series(labels).value_counts().sort_index()
summary_df = pd.DataFrame({"class_name": CLASS_NAMES, "samples_trainval": class_counts.values})
display(summary_df.head())

plt.figure(figsize=(14, 4))
plt.bar(range(NUM_CLASSES), summary_df["samples_trainval"].values)
plt.title("Liczność klas — trainval")
plt.xlabel("Klasa")
plt.ylabel("Liczba próbek")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 8))
for i in range(12):
    img, label = trainval_raw[i]
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(CLASS_NAMES[label], fontsize=9)
    plt.axis("off")
plt.suptitle("Przykładowe obrazy")
plt.tight_layout()
plt.show()


In [ ]:
# Transformacje i DataLoadery
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform_base = transforms.Compose(
    [
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

train_transform_aug = transforms.Compose(
    [
        transforms.Resize(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)


def build_dataloaders(train_transform, batch_size=32, val_ratio=0.2):
    train_ds_aug = datasets.OxfordIIITPet(
        root=DATA_ROOT,
        split="trainval",
        target_types="category",
        transform=train_transform,
        download=False,
    )
    train_ds_eval = datasets.OxfordIIITPet(
        root=DATA_ROOT,
        split="trainval",
        target_types="category",
        transform=eval_transform,
        download=False,
    )
    test_ds = datasets.OxfordIIITPet(
        root=DATA_ROOT,
        split="test",
        target_types="category",
        transform=eval_transform,
        download=False,
    )

    n = len(train_ds_aug)
    indices = torch.randperm(n, generator=torch.Generator().manual_seed(SEED)).tolist()
    split = int((1 - val_ratio) * n)
    train_idx, val_idx = indices[:split], indices[split:]

    train_subset = Subset(train_ds_aug, train_idx)
    val_subset = Subset(train_ds_eval, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    return train_loader, val_loader, test_loader


In [ ]:
# Trening i ewaluacja

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy().tolist()
        y_pred.extend(pred)
        y_true.extend(y.numpy().tolist())
    return y_true, y_pred


def create_optimizer(model, name, lr, weight_decay):
    if name.lower() == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


def train_experiment(config, train_loader, val_loader, test_loader):
    print(f"\n===== {config['title']} =====")
    model = config["model_fn"]().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = create_optimizer(model, config["optimizer"], config["lr"], config.get("weight_decay", 0.0))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = deepcopy(model.state_dict())

    start = time.time()
    for epoch in range(1, config["epochs"] + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        scheduler.step(val_acc)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:02d}/{config['epochs']} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    total_time = time.time() - start

    model.load_state_dict(best_state)
    test_loss, test_acc = run_epoch(model, test_loader, criterion)

    exp_dir = ARTIFACTS_ROOT / config["name"]
    exp_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "num_classes": NUM_CLASSES,
            "class_names": CLASS_NAMES,
            "model_key": config["model_key"],
            "image_size": IMAGE_SIZE,
        },
        exp_dir / "best_model.pth",
    )

    metrics = {
        "train_acc_last": history["train_acc"][-1],
        "val_acc_best": best_val_acc,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "training_time_sec": total_time,
    }

    with open(exp_dir / "history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    with open(exp_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    return {
        "name": config["name"],
        "title": config["title"],
        "model_key": config["model_key"],
        "changes": config["changes"],
        "history": history,
        "metrics": metrics,
    }


In [ ]:
# 3 eksperymenty: bazowy -> ulepszenie -> ulepszenie poprzedniego
experiments = [
    {
        "name": "exp1_baseline",
        "title": "Eksperyment 1: Bazowa CNN",
        "model_key": "baseline",
        "model_fn": lambda: BaselineCNN(NUM_CLASSES),
        "train_transform": train_transform_base,
        "batch_size": 32,
        "epochs": 8,
        "lr": 1e-3,
        "optimizer": "adam",
        "weight_decay": 0.0,
        "changes": [
            "Prosta sieć CNN",
            "Brak Data Augmentation",
            "Punkt odniesienia",
        ],
    },
    {
        "name": "exp2_improved",
        "title": "Eksperyment 2: Ulepszona CNN",
        "model_key": "improved",
        "model_fn": lambda: ImprovedCNN(NUM_CLASSES),
        "train_transform": train_transform_aug,
        "batch_size": 64,
        "epochs": 10,
        "lr": 7e-4,
        "optimizer": "adam",
        "weight_decay": 1e-4,
        "changes": [
            "Dodano BatchNorm",
            "Dodano Dropout",
            "Dodano Data Augmentation",
            "Lepsze hiperparametry",
        ],
    },
    {
        "name": "exp3_transfer",
        "title": "Eksperyment 3: Transfer Learning (ResNet18)",
        "model_key": "transfer",
        "model_fn": lambda: create_transfer_model(NUM_CLASSES, freeze_backbone=False),
        "train_transform": train_transform_aug,
        "batch_size": 64,
        "epochs": 8,
        "lr": 2e-4,
        "optimizer": "adam",
        "weight_decay": 1e-4,
        "changes": [
            "Start od pipeline exp2 (augmentacja)",
            "Pretrained ResNet18",
            "Fine-tuning całego modelu z mniejszym lr",
        ],
    },
]

results = []
for config in experiments:
    train_loader, val_loader, test_loader = build_dataloaders(
        config["train_transform"],
        batch_size=config["batch_size"],
    )
    results.append(train_experiment(config, train_loader, val_loader, test_loader))

print("Trening zakończony.")


In [ ]:
# Wyniki numeryczne + zapis pod frontend
rows = []
for r in results:
    rows.append(
        {
            "experiment": r["name"],
            "title": r["title"],
            "train_acc_last": r["metrics"]["train_acc_last"],
            "val_acc_best": r["metrics"]["val_acc_best"],
            "test_acc": r["metrics"]["test_acc"],
            "test_loss": r["metrics"]["test_loss"],
            "training_time_sec": r["metrics"]["training_time_sec"],
        }
    )

results_df = pd.DataFrame(rows).sort_values("test_acc", ascending=False).reset_index(drop=True)
display(results_df)

with open(ARTIFACTS_ROOT / "summary.json", "w", encoding="utf-8") as f:
    json.dump({"class_names": CLASS_NAMES, "experiments": results}, f, ensure_ascii=False, indent=2)

print("Zapisano artifacts_v2/summary.json")


In [ ]:
# Wizualizacje wyników
plt.figure(figsize=(12, 5))
for r in results:
    plt.plot(r["history"]["val_acc"], label=f"{r['name']} val_acc")
plt.title("Val accuracy — porównanie eksperymentów")
plt.xlabel("Epoka")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(12, 5))
for r in results:
    plt.plot(r["history"]["val_loss"], label=f"{r['name']} val_loss")
plt.title("Val loss — porównanie eksperymentów")
plt.xlabel("Epoka")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(results_df["experiment"], results_df["test_acc"])
plt.title("Test accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# Confusion matrix najlepszego modelu
best_exp_name = results_df.iloc[0]["experiment"]
best_result = next(r for r in results if r["name"] == best_exp_name)
ckpt = torch.load(ARTIFACTS_ROOT / best_exp_name / "best_model.pth", map_location=DEVICE)

best_model = build_model_from_key(ckpt["model_key"], NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(ckpt["state_dict"])
best_model.eval()

_, _, test_loader_eval = build_dataloaders(train_transform_base, batch_size=64)
y_true, y_pred = collect_predictions(best_model, test_loader_eval)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(include_values=False, cmap="Blues", ax=plt.gca(), colorbar=False)
plt.title(f"Confusion matrix — {best_exp_name}")
plt.show()

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
report_df = pd.DataFrame(report).transpose()
display(report_df.head(10))


In [ ]:
# Przykładowe predykcje

def predict_pil(model, pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    idx = int(np.argmax(probs))
    return idx, float(probs[idx])

test_vis = datasets.OxfordIIITPet(root=DATA_ROOT, split="test", target_types="category", download=False)

plt.figure(figsize=(14, 8))
for i in range(12):
    img, label = test_vis[i]
    pred_idx, pred_prob = predict_pil(best_model, img)

    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(
        f"True: {CLASS_NAMES[label]}\nPred: {CLASS_NAMES[pred_idx]} ({pred_prob:.2f})",
        fontsize=8,
    )
    plt.axis("off")

plt.suptitle(f"Predykcje — najlepszy model: {best_exp_name}")
plt.tight_layout()
plt.show()


In [ ]:
# Wnioski
best = results_df.iloc[0]
worst = results_df.iloc[-1]

print("WNIOSKI")
print("-" * 60)
print(f"Najlepszy model: {best['experiment']} (test_acc={best['test_acc']:.4f})")
print(f"Najsłabszy model: {worst['experiment']} (test_acc={worst['test_acc']:.4f})")
print("\nCo można poprawić dalej?")
print("1) więcej epok i cosine scheduler")
print("2) mocniejsza augmentacja (np. RandomErasing)")
print("3) early stopping")
print("4) test innych backbone'ów transfer learning")


## Frontend (v2)

Po treningu uruchom:

```bash
pip install -r requirements.txt
streamlit run frontend_app_v2.py
```

Frontend pokaże:
- opis eksperymentów,
- metryki i wykresy,
- test wytrenowanych modeli na własnych obrazach.
